In [1]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

In [2]:
llm_model = "gpt-4o-mini"

file = 'data/OutdoorClothingCatalog_1000.csv'

In [3]:
import csv
from langchain_core.documents import Document

docs = []
with open(file=file, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        content = "\n".join(f"{k}: {v}" for k, v in row.items())
        docs.append(Document(page_content=content, metadata={"row": i, "source": file}))

In [ ]:
# 
# DEPRECATED: The following code is commented out because the `VectorstoreIndexCreator` and `DocArrayInMemorySearch` are no longer used in the current implementation. 
# Instead, we are using `InMemoryVectorStore` for vector storage and retrieval.
# from langchain.indexes import VectorstoreIndexCreator

# #pip install docarray

# index = VectorstoreIndexCreator(
#     vectorstore_cls=DocArrayInMemorySearch
# ).from_loaders([loader])

# query ="Please list all your shirts with sun protection \
# in a table in markdown and summarize each one."

# llm_replacement_model = OpenAI(temperature=0, 
#                                model='gpt-3.5-turbo-instruct')

# response = index.query(query, 
#                        llm = llm_replacement_model)

# display(Markdown(response))



In [22]:
#!pip install -U langchain-chroma langchain-openai langchain-core chromadb


from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from IPython.display import Markdown, display
from langchain_core.vectorstores import InMemoryVectorStore

In [6]:
#!pip install docarray langchain-community langchain-openai

#!pip uninstall -y openai langchain-openai
#!pip install -U openai langchain-openai

#!pip uninstall -y openai langchain-openai

#!pip cache purge

#!pip install --no-cache-dir -U openai langchain-openai

#!pip show openai

In [12]:
# 1. Construir el vectorstore directamente desde el loader
embeddings = OpenAIEmbeddings()

vectorstore = InMemoryVectorStore.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

In [13]:
# 2. Prompt explícito (antes esto vivía escondido dentro de index.query)
prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:

{context}

Question: {question}"""
)

In [16]:
# 3. Modelo de chat (gpt-3.5-turbo-instruct es un modelo de completions legacy;
#    si quieres seguir usándolo tal cual, usa langchain_openai.OpenAI en vez de ChatOpenAI)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [17]:
# 4. Cadena LCEL equivalente a index.query()
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

In [18]:
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [19]:
# 5. Ejecutar
query = (
    "Please list all your shirts with sun protection "
    "in a table in markdown and summarize each one."
)

In [20]:
response = chain.invoke(query)
display(Markdown(response))

Here is a table summarizing the shirts with sun protection:

| Name                                      | Description Summary                                                                                          |
|-------------------------------------------|--------------------------------------------------------------------------------------------------------------|
| Men's Tropical Plaid Short-Sleeve Shirt   | Lightweight, UPF 50+ rated for sun protection, made of 100% polyester, wrinkle-resistant, with cape venting and two front pockets. |
| Men's Plaid Tropic Shirt, Short-Sleeve    | Designed for fishing, UPF 50+ coverage, made of 52% polyester and 48% nylon, wrinkle-free, evaporates perspiration, with cape venting and two front pockets. |
| Men's TropicVibe Shirt, Short-Sleeve      | Lightweight, UPF 50+ rated, made of 71% nylon and 29% polyester, wrinkle-resistant, with cape venting and two front pockets. |
| Sun Shield Shirt                          | Slightly fitted, UPF 50+ rated, made of 78% nylon and 22% Lycra Xtra Life, moisture-wicking, abrasion resistant, recommended by The Skin Cancer Foundation. |